# Token Merging (ToMe) Ablation Study for YOLOv12 Segmentation

This notebook implements Token Merging (ToMe) from [Token Merging: Your ViT But Faster](https://arxiv.org/abs/2210.09461) for YOLOv12 segmentation models.

## Objectives
- Apply Token Merging to optimal PPC 5×5 model
- Compare performance with baseline (PPC 7×7) and optimal (PPC 5×5)
- Evaluate different merge ratios (0.1, 0.2, 0.25, 0.3)
- Measure FPS, latency, accuracy, and memory usage

## Reference
- Paper: [Token Merging: Your ViT But Faster](https://arxiv.org/abs/2210.09461)
- GitHub: https://github.com/facebookresearch/ToMe
- Key Finding: ToMe can 2× throughput with only 0.2-0.3% accuracy drop

In [3]:
# Setup: imports and helper functions for ToMe placement experiments

import os
from pathlib import Path
import gc
import torch
import pandas as pd

from ultralytics import YOLO
from ultralytics.nn.modules.block import Attention
from token_merging_ablation import TokenMergingWrapper, benchmark_model

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Paths and data configuration
BASELINE_PT = Path('/home/tiehangz/proj/yolov12/model/yolov12n-seg.pt')
OPTIMAL_PT = Path('/home/tiehangz/proj/yolov12/modification/yolov12n-seg-ppc5x5.pt')
# Use a larger COCO subset for more robust evaluation; adjust if you have a custom COCO-seg YAML
TEST_DATA = 'coco.yaml'
IMG_SIZE = 640


def load_optimal_model():
    """Load the optimal PPC 5x5 YOLOv12 segmentation model on the current device."""
    m = YOLO(str(OPTIMAL_PT))
    m.to(device)
    m.eval()
    return m


def get_detect_idx(yolo_model):
    """Return index of the final Detect/Segment head in the underlying nn.Sequential graph."""
    # YOLO wrapper -> task model (e.g. SegmentationModel)
    core = yolo_model.model if hasattr(yolo_model, 'model') else yolo_model
    # Underlying sequential graph lives in `core.model` for Detection/SegmentationModel
    seq = core.model if hasattr(core, 'model') else core

    detect_idx = None
    for m in seq.modules():
        t_attr = getattr(m, 'type', None)
        m_type = t_attr.lower() if isinstance(t_attr, str) else ''
        if m_type in ('segment', 'detect', 'pose', 'obb'):
            detect_idx = getattr(m, 'i', detect_idx)
    return detect_idx


def collect_attention_modules(yolo_model):
    """Collect (name, module, index) for all Attention blocks in the model graph."""
    core = yolo_model.model if hasattr(yolo_model, 'model') else yolo_model
    seq = core.model if hasattr(core, 'model') else core

    modules = []
    for name, module in seq.named_modules():
        if isinstance(module, Attention):
            idx = getattr(module, 'i', None)
            modules.append((name, module, idx))
    modules.sort(key=lambda t: (9999 if t[2] is None else t[2], t[0]))
    return modules


def summarize_attention_layout(yolo_model):
    """Print a summary of where Attention blocks live (indices) for manual inspection."""
    detect_idx = get_detect_idx(yolo_model)
    attns = collect_attention_modules(yolo_model)
    print(f"Detect/Segment head index: {detect_idx}")
    print("Attention modules (name, idx):")
    for name, _, idx in attns:
        print(f"  - {name:30s} idx={idx}")
    return attns, detect_idx


def apply_tome_with_placement(yolo_model, merge_ratio=0.25, placement='all'):
    """Wrap Attention blocks with TokenMergingWrapper according to placement strategy.

    Args:
        yolo_model: Ultralytics YOLO model instance (YOLO class).
        merge_ratio (float): Fraction of tokens to merge (0.0-1.0).
        placement (str): One of
            - 'all': apply ToMe to all Attention modules
            - 'backbone': earlier half of Attention indices (heuristic backbone)
            - 'neck_pre_fpn': later half before head (heuristic neck before FPN fusion)
            - 'after_patch_embed': only the earliest Attention block (right after patch embedding)
    """
    from ultralytics.nn.modules.block import Attention  # ensure correct class binding

    attns, detect_idx = summarize_attention_layout(yolo_model)
    idxs = [idx for _, _, idx in attns if idx is not None and detect_idx is not None and idx < detect_idx]

    if idxs:
        min_idx, max_idx = min(idxs), max(idxs)
        mid_idx = (min_idx + max_idx) / 2.0
    else:
        min_idx = max_idx = mid_idx = None

    def should_wrap(idx: int | None) -> bool:
        if placement == 'all' or idx is None or detect_idx is None:
            return True
        if placement == 'backbone' and mid_idx is not None:
            return idx < mid_idx
        if placement == 'neck_pre_fpn' and mid_idx is not None:
            return mid_idx <= idx < detect_idx
        if placement == 'after_patch_embed' and min_idx is not None:
            # Only the earliest attention block
            return idx == min_idx
        # Fallback: apply everywhere
        return True

    replaced = 0

    def _replace(module, name_prefix=''):
        nonlocal replaced
        for child_name, child_module in list(module.named_children()):
            full_name = f"{name_prefix}.{child_name}" if name_prefix else child_name
            if isinstance(child_module, Attention):
                idx = getattr(child_module, 'i', None)
                if should_wrap(idx):
                    wrapped = TokenMergingWrapper(child_module, merge_ratio=merge_ratio)
                    setattr(module, child_name, wrapped)
                    replaced += 1
                    print(
                        f"  Applied ToMe (ratio={merge_ratio}) to {full_name} "
                        f"(idx={idx}, placement={placement})"
                    )
            else:
                # Recurse into nested modules
                if any(child_module.children()):
                    _replace(child_module, full_name)

    if hasattr(yolo_model, 'model'):
        _replace(yolo_model.model)
    else:
        _replace(yolo_model)

    print(f"Total wrapped attention modules: {replaced}")
    return replaced



Device: cuda


In [4]:
# Run ToMe placement comparison on the optimal PPC 5×5 model

merge_ratio = 0.25
placements = ['none', 'all', 'backbone', 'neck_pre_fpn', 'after_patch_embed']

results = []

# Optional: load previous results to avoid re-running expensive validation
cache_path = Path('/home/tiehangz/proj/yolov12/modification/outputs/tome_placement_ablation_results.csv')
if cache_path.exists():
    cached_df = pd.read_csv(cache_path)
    print(f"Loaded cached results from {cache_path}")
else:
    cached_df = pd.DataFrame()


def get_cached_row(df, placement, merge_ratio):
    if df.empty:
        return None
    mask = (df['placement'] == placement) & (df['merge_ratio'] == merge_ratio)
    if mask.any():
        return df[mask].iloc[0].to_dict()
    return None


# 1) Baseline (no ToMe)
print("\n=== Baseline (no ToMe) ===")
row = get_cached_row(cached_df, 'none', 0.0)
if row is not None:
    print("  Using cached baseline result.")
    results.append(row)
else:
    model = load_optimal_model()
    bench = benchmark_model(
        model,
        test_data=TEST_DATA,
        imgsz=IMG_SIZE,
        warmup=10,
        iterations=100,
        device=str(device),
    )
    row = {
        'placement': 'none',
        'merge_ratio': 0.0,
        'model_path': str(OPTIMAL_PT),
        **bench,
    }
    results.append(row)
    print(f"  FPS={bench['fps']:.2f}, Latency={bench['avg_latency_ms']:.2f} ms, mAP50-95={bench['map50_95']:.4f}")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 2) ToMe placements
for placement in ['all', 'backbone', 'neck_pre_fpn', 'after_patch_embed']:
    print(f"\n=== ToMe placement: {placement} (merge_ratio={merge_ratio}) ===")
    row = get_cached_row(cached_df, placement, merge_ratio)
    if row is not None:
        print("  Using cached result.")
        results.append(row)
        continue

    model = load_optimal_model()
    apply_tome_with_placement(model, merge_ratio=merge_ratio, placement=placement)

    bench = benchmark_model(
        model,
        test_data=TEST_DATA,
        imgsz=IMG_SIZE,
        warmup=10,
        iterations=100,
        device=str(device),
    )
    row = {
        'placement': placement,
        'merge_ratio': merge_ratio,
        'model_path': str(OPTIMAL_PT),
        **bench,
    }
    results.append(row)
    print(f"  FPS={bench['fps']:.2f}, Latency={bench['avg_latency_ms']:.2f} ms, mAP50-95={bench['map50_95']:.4f}")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Summarize and save
results_df = pd.DataFrame(results)
display(results_df[['placement', 'merge_ratio', 'fps', 'avg_latency_ms', 'map50_95']])

output_path = Path('/home/tiehangz/proj/yolov12/modification/outputs/tome_placement_ablation_results.csv')
results_df.to_csv(output_path, index=False)
print(f"\nSaved ToMe placement results to: {output_path}")




=== Baseline (no ToMe) ===
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.690851211547852. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.690851211547852. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.690851211547852. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.690851211547852. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.690851211547852. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.690851211547852. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.690851211547852. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 4.690851211547852. Dividing input by 255.
WARNING ⚠️ torch.Ten

val: Scanning /home/tiehangz/proj/datasets/..datasets/coco/labels/val2017.cache... 4952 images, 48 backgrounds, 0 corrupt: 100%|██████████| 5000/5000 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 313/313 [06:37<00:00,  1.27s/it]


                   all       5000      36335      0.668      0.501       0.55      0.397      0.662      0.484      0.523      0.329
Speed: 0.2ms preprocess, 1.3ms inference, 0.0ms loss, 0.3ms postprocess per image
Saving /home/tiehangz/proj/yolov12/runs/segment/val3/predictions.json...

Evaluating pycocotools mAP using /home/tiehangz/proj/yolov12/runs/segment/val3/predictions.json and /home/tiehangz/proj/datasets/..datasets/coco/annotations/instances_val2017.json...
loading annotations into memory...
Done (t=0.17s)
creating index...
index created!
Loading and preparing results...
DONE (t=4.66s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=18.85s).
Accumulating evaluation results...
DONE (t=3.73s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.399
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.555
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] 

val: Scanning /home/tiehangz/proj/datasets/..datasets/coco/labels/val2017.cache... 4952 images, 48 backgrounds, 0 corrupt: 100%|██████████| 5000/5000 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 313/313 [06:38<00:00,  1.27s/it]


                   all       5000      36335      0.668      0.501       0.55      0.397      0.662      0.484      0.523      0.329
Speed: 0.2ms preprocess, 1.3ms inference, 0.0ms loss, 0.3ms postprocess per image
Saving /home/tiehangz/proj/yolov12/runs/segment/val4/predictions.json...

Evaluating pycocotools mAP using /home/tiehangz/proj/yolov12/runs/segment/val4/predictions.json and /home/tiehangz/proj/datasets/..datasets/coco/annotations/instances_val2017.json...
loading annotations into memory...
Done (t=0.15s)
creating index...
index created!
Loading and preparing results...
DONE (t=4.96s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=18.87s).
Accumulating evaluation results...
DONE (t=3.75s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.399
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.555
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] 

...

                   all       5000      36335      0.668      0.501       0.55      0.397      0.662      0.484      0.523      0.329
Speed: 0.2ms preprocess, 1.3ms inference, 0.0ms loss, 0.3ms postprocess per image
Saving /home/tiehangz/proj/yolov12/runs/segment/val6/predictions.json...

Evaluating pycocotools mAP using /home/tiehangz/proj/yolov12/runs/segment/val6/predictions.json and /home/tiehangz/proj/datasets/..datasets/coco/annotations/instances_val2017.json...
loading annotations into memory...
Done (t=0.15s)
creating index...
index created!
Loading and preparing results...
DONE (t=5.03s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=18.89s).
Accumulating evaluation results...
DONE (t=3.69s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.399
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.555
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] 

val: Scanning /home/tiehangz/proj/datasets/..datasets/coco/labels/val2017.cache... 4952 images, 48 backgrounds, 0 corrupt: 100%|██████████| 5000/5000 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 313/313 [06:42<00:00,  1.28s/it]


                   all       5000      36335      0.668      0.501       0.55      0.397      0.662      0.484      0.523      0.329
Speed: 0.2ms preprocess, 1.3ms inference, 0.0ms loss, 0.3ms postprocess per image
Saving /home/tiehangz/proj/yolov12/runs/segment/val7/predictions.json...

Evaluating pycocotools mAP using /home/tiehangz/proj/yolov12/runs/segment/val7/predictions.json and /home/tiehangz/proj/datasets/..datasets/coco/annotations/instances_val2017.json...
loading annotations into memory...
Done (t=0.15s)
creating index...
index created!
Loading and preparing results...
DONE (t=3.71s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=20.33s).
Accumulating evaluation results...
DONE (t=3.74s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.399
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.555
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] 

,placement,merge_ratio,fps,avg_latency_ms,map50_95
0,none,0.00,161.655189,6.186006,0.3292
1,all,0.25,152.697215,6.548908,0.3292
2,backbone,0.25,164.512192,6.078577,0.3292
3,neck_pre_fpn,0.25,159.866823,6.255207,0.3292
4,after_patch_embed,0.25,170.898923,5.851412,0.3292



Saved ToMe placement results to: /home/tiehangz/proj/yolov12/modification/outputs/tome_placement_ablation_results.csv
